In [1]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Flatten, Dense
from tensorflow.keras.regularizers import l2

In [2]:
# Definicja hiperparametrów
num_words = 5000  # Liczba słów w naszym słowniku
maxlen = 200  # Maksymalna długość tekstu
embedding_dim = 16 # Wielkość wektora embeddingu (hiperparametr modelu)
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words = num_words) # Pobieramy dane
str(x_train[0])

    # '[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 2, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 2, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 2, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 2, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 2, 19, 178, 32]'

17464789/17464789 [==============================] - 1s 0us/step


'[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 2, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 2, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 2, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 2, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 2, 19, 178, 32]'

In [3]:
def vector_to_text(imdb_vector, label):
    reverse_index = {v:k for k,v in imdb.get_word_index().items()}
    return f"""Comment: {" ".join([reverse_index.get(i-3, "#") for i in imdb_vector])}
            Label: {"Positive" if label == 1 else "Negative"}"""

In [4]:
vector_to_text(x_train[0], y_train[0]), vector_to_text(x_train[1], y_train[1]) , vector_to_text(x_train[2], y_train[2])

1641221/1641221 [==============================] - 1s 0us/step


("Comment: # this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert # is an amazing actor and now the same being director # father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for # and would recommend it to everyone to watch and the fly # was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also # to the two little # that played the # of norman and paul they were just brilliant children are often left out of the # list i think because the stars that play them all grown up are such a big # for the whole film but these children are amazing and should be # for what they have done don't you think the whol

In [5]:
# Robimy padding komentarzy tak, aby wszystekie miały tę sama długość
x_train = pad_sequences(x_train, maxlen=maxlen)
x_test = pad_sequences(x_test, maxlen=maxlen)

In [7]:
## Tworzymy model sieci neuronowej z jedną warstwę ukryta z 16 węzłami (taki mamy rozmiar embeddingu)
def build_keras_model(input_dim, output_dim):
    model = Sequential()
    # Używamy tutaj regularyzacji L2, aby model nam nie overfitował
    model.add(Embedding(input_dim = input_dim, output_dim = output_dim,input_length=maxlen, embeddings_regularizer=l2(0.01)))
    model.add(Flatten())
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model
model = build_keras_model(num_words, embedding_dim)

In [8]:
history = model.fit(x_train, y_train, epochs=10, batch_size=128, validation_data=(x_test, y_test))

Epoch 1/10


196/196 [==============================] - 2s 5ms/step - loss: 0.7286 - accuracy: 0.6020 - val_loss: 0.6368 - val_accuracy: 0.7076
Epoch 2/10
196/196 [==============================] - 1s 4ms/step - loss: 0.5819 - accuracy: 0.7710 - val_loss: 0.5434 - val_accuracy: 0.8017
Epoch 3/10
196/196 [==============================] - 1s 4ms/step - loss: 0.5228 - accuracy: 0.8163 - val_loss: 0.5080 - val_accuracy: 0.8223
Epoch 4/10
196/196 [==============================] - 1s 4ms/step - loss: 0.4926 - accuracy: 0.8326 - val_loss: 0.5016 - val_accuracy: 0.8054
Epoch 5/10
196/196 [==============================] - 1s 4ms/step - loss: 0.4725 - accuracy: 0.8417 - val_loss: 0.4647 - val_accuracy: 0.8433
Epoch 6/10
196/196 [==============================] - 1s 4ms/step - loss: 0.4562 - accuracy: 0.8491 - val_loss: 0.4493 - val_accuracy: 0.8529
Epoch 7/10
196/196 [==============================] - 1s 4ms/step - loss: 0.4454 - accuracy: 0.8508 - val_loss: 0.4419 - val_accuracy: 0.8526
Epoc

In [9]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, 200, 16)           80000     
                                                                 
 flatten_1 (Flatten)         (None, 3200)              0         
                                                                 
 dense_1 (Dense)             (None, 1)                 3201      
                                                                 
Total params: 83201 (325.00 KB)
Trainable params: 83201 (325.00 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [10]:
from sklearn.metrics import classification_report
# Predict the sentiment on the test dataset
y_pred = (model.predict(x_test) > 0.5).astype("int32")
# Print classification report
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

782/782 [==============================] - 1s 964us/step
              precision    recall  f1-score   support

    Negative       0.84      0.89      0.86     12500
    Positive       0.88      0.83      0.85     12500

    accuracy                           0.86     25000
   macro avg       0.86      0.86      0.86     25000
weighted avg       0.86      0.86      0.86     25000



In [11]:
def encode_text(text):
    import re # import biblioteki do wyrazen regularnych
    def get_word_index(word): # Funkcja do zakodowania slowo za pomoca liczby z uwzglednieniem limitu slow
        w_idx = index.get(word, -1)
        return w_idx + 3 if w_idx <= num_words else 2
    index = imdb.get_word_index()
    text = re.sub(r'[^a-z ]', '', text.lower()) # Usuniecie wszystkich znakow, ktore nie sa literami oraz zamiana duzych liter na male
    encoded = [get_word_index(word) for word in text.split(" ")] # Zakodowanie wyrazow
    return pad_sequences([[1] + list(encoded)], maxlen=maxlen) # Dodanie znaku startu i ustawienie dlugosci na 200 słów

Ćwiczenia \/